# Tools
Models can request to call tools that perform tasks such as fetching data from a database, searching the web, or running code. Tools are pairings of:

A schema, including the name of the tool, a description, and/or argument definitions (often a JSON schema)
A function or coroutine to execute.

In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from langchain.chat_models import init_chat_model
model = init_chat_model("groq:qwen/qwen3-32b")

response = model.invoke("Hello! How are you?")
response

AIMessage(content='<think>\nOkay, the user is asking how I\'m doing. I need to respond in a friendly and conversational way. I should mention that I\'m functioning well and ready to help. Also, maybe add a bit of enthusiasm to keep the conversation light. Let me check if there\'s any specific tone the user might expect. Since they said "Hello! How are you?" it\'s probably a casual greeting, so a cheerful and open response should work. I can offer assistance to show I\'m here to help with whatever they need. Make sure the response is concise but not too short. Alright, time to put that together.\n</think>\n\nHello! I\'m doing well, thanks for asking! I\'m all set and ready to help you out. 😊 How can I assist you today?', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 161, 'prompt_tokens': 14, 'total_tokens': 175, 'completion_time': 0.337220794, 'completion_tokens_details': None, 'prompt_time': 0.000794129, 'prompt_tokens_details': None, 'queue_time': 0.022

In [ ]:
from langchain.tools import tool

@tool # decorator needed when passing in with bind_tools method.
def get_weather(location: str)->str:
    """Get the weather at a location """
    return f"Its sunny in {location}."

model_with_tools = model.bind_tools([get_weather])

In [5]:
response = model_with_tools.invoke("Whats the weather like in Boston?")

print(response)

for tool_call in response.tool_calls:
    print(f"Tool: {tool_call["name"]}")
    print(f"Args: {tool_call["args"]}")

content='' additional_kwargs={'reasoning_content': 'Okay, the user is asking about the weather in Boston. I need to use the get_weather function. Let me check the function parameters. It requires a location, which in this case is Boston. I should structure the tool call with the location parameter set to "Boston". Make sure the JSON is correctly formatted. No other parameters are needed. Just call get_weather with location: Boston.\n', 'tool_calls': [{'id': '8mv0vxw4f', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 102, 'prompt_tokens': 153, 'total_tokens': 255, 'completion_time': 0.167578116, 'completion_tokens_details': {'reasoning_tokens': 78}, 'prompt_time': 0.006039336, 'prompt_tokens_details': None, 'queue_time': 0.169149817, 'total_time': 0.173617452}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_2bfcc54d36', 'service_tier': 'on_demand', 'finish_reason': 'tool_

### Tool Execution Steps - how the tool will be called by the llm

In [ ]:
# Step 1: Model generates tool calls
messages = [{"role": "user", "content": "What's the weather in Boston?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# Step 3: Pass results back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)
# "The current weather in Boston is sunny."

The weather in Boston is sunny. Let me know if you need more details! ☀️


In [7]:
messages

[{'role': 'user', 'content': "What's the weather in Boston?"},
 AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for the weather in Boston. I need to use the get_weather function. The function requires a location parameter. Boston is the location here. So I should call get_weather with location set to "Boston". Let me make sure there are no typos. Everything looks good. I\'ll format the tool call as specified.\n', 'tool_calls': [{'id': 'a0y9w2dz3', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 94, 'prompt_tokens': 153, 'total_tokens': 247, 'completion_time': 0.138450055, 'completion_tokens_details': {'reasoning_tokens': 70}, 'prompt_time': 0.006049817, 'prompt_tokens_details': None, 'queue_time': 0.17903626, 'total_time': 0.144499872}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_2bfcc54d36', 'service_tier': 'on_demand', 'finish